# Support Vector Machines

In the notes, we solved a small SVM by hand: we wrote the Lagrangian, went through the cases of
complementary slackness, and read the multipliers $\alpha_k$ off the KKT conditions.

Doing this by hand stops being reasonable at about five points.
In this notebook we let the computer do it, but we do **not** call a ready made `fit` function
straight away: we write the Lagrangian ourselves, hand it to a general purpose solver, and read
the multipliers it returns.

## Generated clouds of points

Step -1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

Step 0: Generate the data

Build two clouds of $20$ points each in the plane:

- the class $+1$ around the point $(3,3)$,
- the class $-1$ around the point $(0,0)$,

both with a standard deviation of $0.8$ (use `rng.normal(loc, scale, size)`).

Then stack them into a single matrix `X` of shape $(40, 2)$ and build the vector of labels `y`,
whose entries are $+1$ or $-1$ (careful: not $0$ and $1$, the whole method needs $\pm 1$).

The seed is fixed so that everybody gets the same picture.

In [ ]:
rng = np.random.default_rng(0)
n = 20

# X_pos = ...
# X_neg = ...

# X = ...   # shape (40, 2)
# y = ...   # shape (40,), values +1 and -1

# X.shape, y.shape

Step 1: Plot the two clouds

Use a different colour (and ideally a different marker) for each class.
Check with your eyes that the two clouds can be separated by a straight line, otherwise the rest
of the notebook has no solution.

## The problem to solve

The SVM looks for the separating line that leaves as much room as possible on both sides:

$$\text{minimize } \ \frac{1}{2}\|\vec{w}\|^2
\qquad \text{subject to} \qquad
y_k \left( \vec{w} \cdot \vec{x}_k - b \right) \geq 1 \quad \text{for every } k$$

The Lagrangian, with one multiplier $\alpha_k \geq 0$ per data point, is

$$\mathcal{L}(\vec{w}, b, \alpha)
= \frac{1}{2}\|\vec{w}\|^2 - \sum_k \alpha_k \left[ y_k \left( \vec{w} \cdot \vec{x}_k - b \right) - 1 \right]$$

Setting its derivatives with respect to $\vec{w}$ and $b$ to zero gives

$$\vec{w} = \sum_k \alpha_k y_k \vec{x}_k
\qquad \text{and} \qquad
\sum_k \alpha_k y_k = 0$$

and putting these two back into $\mathcal{L}$ makes $\vec{w}$ and $b$ disappear completely,
leaving a problem in the multipliers alone:

$$\text{maximize } \ W(\alpha) = \sum_k \alpha_k - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j \, y_i y_j \, \vec{x}_i \cdot \vec{x}_j
\qquad \text{subject to} \qquad
\sum_k \alpha_k y_k = 0, \quad \alpha_k \geq 0$$

This is the **dual problem**, and it is the one every SVM library actually solves: the unknowns
are the multipliers, and the constraints are as simple as constraints get.

Step 2: Build the matrix of the dual

With $H_{ij} = y_i y_j \, \vec{x}_i \cdot \vec{x}_j$, the function to maximize is

$$W(\alpha) = \sum_k \alpha_k - \frac{1}{2} \, \alpha^T H \alpha$$

Compute the matrix of all the scalar products $\vec{x}_i \cdot \vec{x}_j$ in one line
(it is called the *Gram matrix*), then multiply it entry by entry by $y_i y_j$
(`np.outer` is useful here).

In [ ]:
# K = ...   # Gram matrix, shape (40, 40)
# H = ...   # H[i,j] = y_i y_j x_i . x_j

# H.shape

Step 3: Write the function to give to the solver

Solvers minimize, so we give them $-W(\alpha)$.
Write it, and write its gradient too:

$$-W(\alpha) = \frac{1}{2} \alpha^T H \alpha - \sum_k \alpha_k
\qquad \qquad
\nabla \left( -W \right)(\alpha) = H \alpha - \mathbb{1}$$

The gradient is optional (the solver can estimate it), but giving it makes the answer much more
precise, and we will need precision to decide which multipliers are zero.

In [ ]:
# def neg_dual(a):
#     ...

# def neg_dual_grad(a):
#     ...

# neg_dual(np.zeros(len(y)))   # should be 0

Step 4: Solve it

We use `scipy.optimize.minimize` with the method `SLSQP`, which is made for exactly this kind of
problem (a smooth objective with constraints).
Three things must be passed:

- the starting point, for instance $\alpha = 0$;
- the sign conditions $\alpha_k \geq 0$, through `bounds=[(0, None)] * len(y)`;
- the equality constraint $\sum_k \alpha_k y_k = 0$, through
  `constraints=[{"type": "eq", "fun": lambda a: a @ y, "jac": lambda a: y}]`.

Add `options={"maxiter": 2000, "ftol": 1e-12}` to ask for a tight answer, and always check
`res.success` before believing anything.

In [ ]:
# res = minimize(...)
# res.success, res.message

Step 5: Read the multipliers

Print them, rounded to three or four decimals.

Most of them should be zero (up to the precision of the solver): those points are the ones
sitting strictly outside the margin band, and they play no role at all.
The few that are not zero are the **support vectors**.

Build a boolean mask `sv = alpha > 1e-6` and count them.

In [ ]:
# alpha = res.x
# print(np.round(alpha, 4))

# sv = ...
# sv.sum()

Step 6: Recover $\vec{w}$ and $b$

The first stationarity condition gives $\vec{w}$ directly:

$$\vec{w} = \sum_k \alpha_k y_k \vec{x}_k$$

For $b$, use any support vector: it satisfies $y_k (\vec{w} \cdot \vec{x}_k - b) = 1$ exactly,
so $b = \vec{w} \cdot \vec{x}_k - y_k$.
Averaging that over all the support vectors is the usual way to reduce the numerical error.

In [ ]:
# w = ...
# b = ...
# w, b

Step 7: Check the answer

Four things are worth checking, and they are all one line each:

1. $\sum_k \alpha_k y_k = 0$ (the constraint we imposed);
2. every point satisfies $y_k \left( \vec{w} \cdot \vec{x}_k - b \right) \geq 1$ (nothing inside the band);
3. the support vectors, and only they, satisfy it with equality;
4. the width of the band is $\frac{2}{\|\vec{w}\|}$.

Step 8: Plot the solution

On top of the two clouds, draw:

- the decision boundary $\vec{w} \cdot \vec{x} - b = 0$;
- the two edges of the band, $\vec{w} \cdot \vec{x} - b = \pm 1$, as dashed lines;
- a big empty circle around each support vector.

To draw the line $w_1 x + w_2 y - b = c$, isolate the second coordinate:
$y = \left( b + c - w_1 x \right) / w_2$.

Step 9: Put it in a function

We will need the same computation again in the second part, so wrap Steps 2 to 6 into a single
function taking `X` and `y` and returning `alpha`, `w`, `b` and the mask of support vectors.

In [ ]:
# def fit_svm(X, y):
#     ...
#     return alpha, w, b, sv

# fit_svm(X, y)

Step 10: Compare with a real library

`sklearn` has an SVM, and it solves the very same dual.
Fit `SVC(kernel="linear", C=1e6)` on the data.
The huge `C` is there to say "no point is allowed inside the band", which is the problem we
solved (we will come back to what a smaller `C` means at the very end).

Then compare:

- `svc.coef_[0]` with your $\vec{w}$;
- `-svc.intercept_[0]` with your $b$ (careful with the sign: `sklearn` writes the model as
  $\vec{w} \cdot \vec{x} + \text{intercept}$, while we write $\vec{w} \cdot \vec{x} - b$);
- `svc.support_` with the indices of your support vectors;
- `np.abs(svc.dual_coef_[0])` with your non-zero multipliers (`dual_coef_` stores $\alpha_k y_k$).

In [ ]:
# from sklearn.svm import SVC

Step 11: Two experiments

1. Move one of the points that is **not** a support vector, anywhere you like as long as it stays
   on its own side of the band, and solve again. What changes?
2. Move one of the **support vectors** a little. What changes now?

---

## Loan application

A bank has kept the record of $60$ past loan applications that were all accepted.
For each one it knows:

| column | meaning |
|---|---|
| `income` | annual income of the applicant, in thousands of euros |
| `years_employed` | number of years in the current job |
| `repaid` | $+1$ if the loan was repaid, $-1$ if the applicant defaulted |

The bank would like a simple, explainable rule on these two numbers to decide about the next
applicant, and it wants that rule to be as far as possible from the files it has already seen:
this is exactly a maximum margin problem.

The data is in `loans.csv`, next to this notebook
(or from GitHub: https://github.com/pauldubois98/RefresherMaths2026/blob/main/SessionBinaryClassification/loans.csv).

Step 0: Read the data, and look at it

In [ ]:
import pandas as pd

df = pd.read_csv('loans.csv')
df.head()

How many applicants of each class are there?

Step 1: Build `X` (shape $(60,2)$) and `y` (values $\pm 1$)

Step 2: Plot the data, one colour per class

Are the two classes separable by a straight line?

Step 3: Standardize the features

Subtract the mean and divide by the standard deviation of each column, exactly as in the PCA
session, and keep `mu` and `sigma` aside: we will need them to translate the answer back into
euros and years at the end.

Why is this necessary here?
Look at the two columns: one is measured in thousands of euros and runs up to $90$, the other is
a number of years and runs up to $20$.
The quantity $\|\vec{w}\|^2$ that the SVM minimizes adds the two together, so without
standardizing, the margin would be measured in a unit that mixes euros and years, and the feature
with the larger numbers would dominate the answer for no good reason.

In [ ]:
# mu = ...
# sigma = ...
# Z = ...

Step 4: Solve the SVM on the standardized data with your `fit_svm` function

Step 5: Who are the support vectors?

Print the rows of the dataframe corresponding to the support vectors.

These are the borderline files: the ones close enough to the decision to matter.
The bank's rule depends only on them, and the other applications, however many they are, could be
thrown away without changing anything.

Step 6: Translate the rule back into euros and years

We solved the problem on the standardized data $\vec{z} = \left( \vec{x} - \mu \right) / \sigma$,
so the rule is $\vec{w} \cdot \vec{z} - b > 0$.
Rewriting it in terms of the original $\vec{x}$:

$$\vec{w} \cdot \frac{\vec{x} - \mu}{\sigma} - b
= \underbrace{\left( \frac{w_1}{\sigma_1}, \frac{w_2}{\sigma_2} \right)}_{\vec{w}_\text{orig}} \cdot \vec{x}
- \underbrace{\left( b + \frac{w_1 \mu_1}{\sigma_1} + \frac{w_2 \mu_2}{\sigma_2} \right)}_{b_\text{orig}}$$

Compute $\vec{w}_\text{orig}$ and $b_\text{orig}$, and write the rule the bank should use as a
sentence, of the form:

> lend if $\ \ldots \times \text{income} + \ldots \times \text{years} > \ldots$

Does the sign of each coefficient make sense?

Step 7: Plot the data again, in the original units, with the boundary and the two edges of the band

Step 8: Three new applicants show up

| | income | years employed |
|---|---|---|
| A | $40$ | $5$ |
| B | $55$ | $8$ |
| C | $30$ | $18$ |

Predict the decision for each of them, and give the value of $h(\vec{x})$, not only its sign:
a value close to $0$ means the applicant falls inside the band, that is, a file the rule is not
confident about.

Step 9: A file that ruins everything

Add to the data a new applicant who **defaulted** although their income was $80$ and they had
$15$ years of employment, and solve again.

Look at the largest multiplier (`res.success` is not to be trusted here: the solver may well
report a success while the answer it returns is nonsense).
What is happening, and why could it not have been otherwise?

The hard margin problem we wrote has no solution at all when the two classes overlap: no line
separates them, so no $(\vec{w}, b)$ satisfies the constraints, and the dual has no maximum
either, its value growing without bound.
That is what the multipliers of size $10^{15}$ are telling us.

This is what the **soft margin** is for: a few points are allowed inside the band, at a price,
and the multipliers become bounded by a constant $C$, which is exactly the parameter we set to
`1e6` in Step 10 to switch that tolerance off.

Fit `SVC(kernel="linear", C=1)` on the data with the bad file included, and plot the result.
How many support vectors are there now?

Step 10: To finish

Two questions worth thinking about:

1. The bank wants to add a third feature (the amount asked for). Nothing in what we wrote uses the
   fact that the points live in the plane. What would need to change in the code, and what would
   be lost?
2. Our rule treats the two mistakes symmetrically. For a bank, refusing a good client and
   accepting a bad one do not cost the same. Where, in the problem we wrote, could that asymmetry
   be introduced?